# HealthConnect No-Show Prediction — Week 5 Pipeline Walkthrough
## Machine Learning Engineering Track

**Project:** HealthConnect Clinic — Improving Patient Appointment Attendance and Healthcare Support Using Data and AI
**Analyst:** Raji Toluwanimi Samuel, Junior Machine Learning Engineer
**Week:** 5 — ML Pipeline Development & Initial Implementation

---
This notebook demonstrates the pipeline implemented in `src/` end-to-end: data ingestion, validation, point-in-time feature engineering, a time-based train/test split, and a baseline model trained through the `NoShowRiskModel` wrapper. It is a **demonstration and documentation notebook** — the actual reusable logic lives in the `src/` package and is covered by the automated tests in `tests/`, not in this notebook.

This is implementation work, not final modelling: the Data Science track owns the target model choice and feature set. This notebook proves the *system* those decisions will plug into actually works.


In [1]:
import sys
sys.path.insert(0, '..')  # so `src` is importable from notebooks/

import pandas as pd
from src import config, data_ingestion, data_validation, feature_engineering
from src.model import NoShowRiskModel, time_based_split
from src.pipeline import run_pipeline

pd.set_option('display.max_columns', 40)

## 1. Week 4 Foundation — Brief Recap

Week 4 defined the system architecture for this pipeline: ingestion → validation → point-in-time feature engineering → versioned feature store → training → model registry → batch scoring → risk score output, with a monitoring/retraining feedback loop. Two Week 4 findings directly shaped this week's implementation:

- `waiting_time_minutes` was flagged as a likely leakage risk (populated even for appointments that were never attended).
- The prediction target was defined as No-Show vs Attended, with `Cancelled` excluded from the primary binary classifier by default.

Week 5 goal: turn that design into working, tested code.


## 2. Data Ingestion

`data_ingestion.load_raw_data()` loads the raw CSV without ever modifying it, and fails loudly (via `DataIngestionError`) if the file is missing, empty, or missing an expected column — rather than silently proceeding with bad data.

In [2]:
raw_df = data_ingestion.load_raw_data()
print(f"Loaded {raw_df.shape[0]} rows, {raw_df.shape[1]} columns")
raw_df.head()

Loaded 5000 rows, 18 columns


,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show


## 3. Data Validation

`data_validation.run_all_checks()` runs every automated check and returns a structured report. Two of these checks directly confirm Week 4 suspicions and drove this week's engineering decisions.

In [3]:
validation_report = data_validation.run_all_checks(raw_df)
validation_report

{'row_count': 5000,
 'column_count': 18,
 'missing_values': {'reminder_channel': {'missing_count': 1366,
   'missing_pct': 27.32},
  'distance_to_clinic_km': {'missing_count': 90, 'missing_pct': 1.8},
  'waiting_time_minutes': {'missing_count': 60, 'missing_pct': 1.2}},
 'duplicates': {'duplicate_rows': 0, 'duplicate_appointment_ids': 0},
 'waiting_time_leakage': {'checked': True,
  'not_attended_rows': 2686,
  'rows_with_unexpected_waiting_time': 2647,
  'leakage_confirmed': True},
 'historical_field_consistency': {'checked': True,
  'patients_checked': 1696,
  'patients_chronologically_consistent': 701,
  'consistency_rate_pct': 41.3,
  'conclusion': 'Raw previous_appointments/previous_no_shows are not reliably point-in-time correct -- pipeline recomputes these fields instead of using them as provided.'}}

### 3.1 Leakage check — confirmed

`waiting_time_minutes` is non-null for **2,647 of 2,686** appointments that were not actually attended. A patient cannot accumulate a clinic wait time for a visit that never happened — this confirms the Week 4 suspicion as a hard fact, and `feature_engineering.drop_leakage_columns()` removes this column unconditionally.

### 3.2 Historical field consistency check — new Week 5 finding

Only **41.3%** of patients have a chronologically consistent `previous_appointments` count (i.e. it doesn't always increase as their appointments proceed through time). This was discovered *during* Week 5 implementation, not anticipated in Week 4 — per the brief's instruction to document and justify decisions that change based on new analysis, this finding is why the pipeline recomputes these fields from scratch (Section 4) instead of trusting the raw columns.


In [4]:
# Illustrating the inconsistency directly on one real patient
example_patient = raw_df[raw_df['patient_id'] == raw_df['patient_id'].iloc[0]]
example_patient[['patient_id', 'appointment_date', 'previous_appointments', 'previous_no_shows', 'appointment_outcome']].sort_values('appointment_date')

,patient_id,appointment_date,previous_appointments,previous_no_shows,appointment_outcome
0,P-1613,2/18/2025,2,0,No-Show
4058,P-1613,4/23/2025,7,0,Attended


## 4. Feature Engineering

### 4.1 Recomputing point-in-time historical features

`feature_engineering.recompute_historical_features()` sorts each patient's appointments chronologically and counts, for every appointment, only the *earlier* appointments and no-shows that patient actually had — producing `computed_previous_appointments` and `computed_previous_no_shows` that are leakage-free by construction, unlike the raw columns.

In [5]:
with_history = feature_engineering.recompute_historical_features(raw_df)
cols = ['patient_id', 'appointment_date', 'previous_appointments', 'computed_previous_appointments',
        'previous_no_shows', 'computed_previous_no_shows', 'appointment_outcome']
with_history[with_history['patient_id'] == raw_df['patient_id'].iloc[0]][cols].sort_values('appointment_date')

,patient_id,appointment_date,previous_appointments,computed_previous_appointments,previous_no_shows,computed_previous_no_shows,appointment_outcome
4467,P-1613,2/18/2025,2,0,0,0,No-Show
4468,P-1613,4/23/2025,7,1,0,1,Attended


Notice the recomputed columns always start at 0 for a patient's first appointment and increase only after a *prior* appointment — exactly the point-in-time correctness the raw columns lacked.

### 4.2 Full feature pipeline

`feature_engineering.run_feature_pipeline()` runs the complete sequence: recompute history → build target (excluding `Cancelled`) → impute missing values → engineer `booking_lead_bucket` → drop leakage columns → drop identifiers → encode categoricals.

In [6]:
processed_df, validation_report = run_pipeline(save_output=True)
print(f"Processed shape: {processed_df.shape}")
print(f"Rows excluded (Cancelled): {len(raw_df) - len(processed_df)}")
processed_df.head()

2026-09-03 08:46:23,521 [INFO] Step 1/3: Ingesting raw data


2026-09-03 08:46:23,544 [INFO] Loaded 5000 rows, 18 columns


2026-09-03 08:46:23,545 [INFO] Step 2/3: Running data validation checks


2026-09-03 08:46:24,450 [WARNING] Leakage check CONFIRMED: 2647 not-attended rows have a non-null waiting_time_minutes. This column is excluded by feature_engineering.py.


2026-09-03 08:46:24,452 [WARNING] Historical field consistency: only 41.3% of patients have chronologically consistent previous_appointments values. Pipeline recomputes these fields instead of trusting the raw columns.


2026-09-03 08:46:24,452 [INFO] Step 3/3: Running feature engineering


2026-09-03 08:46:25,021 [INFO] Feature engineering complete: 4737 rows, 34 columns (263 rows excluded, consistent with EXCLUDE_LABELS_FROM_TRAINING=['Cancelled'])


2026-09-03 08:46:25,046 [INFO] Saved processed dataset to /home/claude/healthconnect-repo/data/processed/healthconnect_processed.csv


2026-09-03 08:46:25,047 [INFO] Saved validation report to /home/claude/healthconnect-repo/data/processed/data_validation_report.json


Processed shape: (4737, 34)
Rows excluded (Cancelled): 263


,appointment_id,age,appointment_date,booking_lead_days,distance_to_clinic_km,computed_previous_appointments,computed_previous_no_shows,target,gender_Male,gender_Prefer not to say,age_group_25-34,age_group_35-44,age_group_45-54,age_group_55-64,age_group_65+,appointment_type_Follow-up,appointment_type_General Consultation,appointment_type_Specialist Consultation,appointment_day_Monday,appointment_day_Saturday,appointment_day_Sunday,appointment_day_Thursday,appointment_day_Tuesday,appointment_day_Wednesday,appointment_time_Evening,appointment_time_Morning,reminder_sent_Yes,reminder_channel_No Reminder Sent,reminder_channel_SMS,reminder_channel_WhatsApp,booking_lead_bucket_3-7 days,booking_lead_bucket_8-14 days,booking_lead_bucket_15-30 days,booking_lead_bucket_30+ days
0,HC-03844,67,6/13/2025,10,1.3,0,0,0,True,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,False,True,True,False,True,False,False,True,False,False
1,HC-02941,68,1/26/2026,38,7.1,1,0,0,True,False,False,False,False,False,True,False,True,False,True,False,False,False,False,False,False,True,False,True,False,False,False,False,False,True
2,HC-03305,57,3/6/2025,20,10.3,0,0,1,False,False,False,False,False,True,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,True,False
3,HC-04380,59,5/23/2026,44,6.4,1,1,1,True,False,False,False,False,True,False,True,False,False,False,True,False,False,False,False,False,False,True,False,True,False,False,False,False,True
4,HC-02979,35,6/4/2025,28,1.5,0,0,0,False,False,False,True,False,False,False,False,True,False,False,False,False,False,False,True,False,True,True,False,True,False,False,False,True,False


**Row count check:** raw data had 5,000 rows; the processed dataset has fewer because `Cancelled` appointments (5.3% of the data, matching the Week 4 estimate) are excluded from the primary binary classifier by design, not by accident.

**Column check:** `waiting_time_minutes`, `previous_appointments`/`previous_no_shows` (raw), and `patient_id` are all absent from the processed output — the leakage guard and the identifier-exclusion rule are both enforced in code, not just in documentation.


In [7]:
print("Leakage columns present in output:", [c for c in config.LEAKAGE_COLUMNS if c in processed_df.columns] or "None")
print("Raw historical columns present in output:", [c for c in config.RAW_HISTORICAL_COLUMNS if c in processed_df.columns] or "None")
print("patient_id present in output:", "patient_id" in processed_df.columns)

Leakage columns present in output: None
Raw historical columns present in output: None
patient_id present in output: False


## 5. Train/Test Strategy

`model.time_based_split()` splits the processed data by `appointment_date` rather than randomly: the test set is always chronologically after the training set. This matters because the deployed system will only ever predict forward in time — a random split would let the model implicitly train on information from dates after some of its own test cases, which does not reflect how it will actually be used.

In [8]:
train_df, test_df = time_based_split(processed_df)
print(f"Train: {len(train_df)} rows, {train_df['appointment_date'].min()} to {train_df['appointment_date'].max()}")
print(f"Test:  {len(test_df)} rows, {test_df['appointment_date'].min()} to {test_df['appointment_date'].max()}")

Train: 3789 rows, 1/1/2025 to 9/9/2025
Test:  948 rows, 3/13/2026 to 6/9/2026


## 6. Baseline Model

`NoShowRiskModel` wraps a Logistic Regression baseline (`class_weight='balanced'`, since the target is close to evenly split rather than heavily imbalanced) behind a fixed `fit` / `predict` / `predict_proba` / `save` / `load` interface. Any other estimator can be substituted later without changing any calling code — this is the extension point for the Data Science track's eventual model choice.

In [9]:
NON_FEATURE_COLUMNS = ['appointment_id', 'appointment_date', 'target']
feature_columns = [c for c in processed_df.columns if c not in NON_FEATURE_COLUMNS]

X_train, y_train = train_df[feature_columns], train_df['target']
X_test, y_test = test_df[feature_columns], test_df['target']

model = NoShowRiskModel()
model.fit(X_train, y_train)
print(f"Model fitted on {len(feature_columns)} features, {len(X_train)} training rows.")

Model fitted on 31 features, 3789 training rows.


## 7. Initial Evaluation

The focus at this stage is establishing a **reliable, working baseline** — not the best-performing model. Final model selection is the Data Science track's responsibility.

In [10]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, confusion_matrix)

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)

metrics = {
    'Accuracy': accuracy_score(y_test, predictions),
    'Precision': precision_score(y_test, predictions),
    'Recall': recall_score(y_test, predictions),
    'F1 Score': f1_score(y_test, predictions),
    'ROC-AUC': roc_auc_score(y_test, probabilities),
}
for k, v in metrics.items():
    print(f"{k:12s}: {v:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

Accuracy    : 0.6245
Precision   : 0.6326
Recall      : 0.6273
F1 Score    : 0.6299
ROC-AUC     : 0.6677

Confusion Matrix:
[[289 176]
 [180 303]]


### Interpretation

The baseline achieves modest but genuine predictive signal (ROC-AUC ~0.67) on a purely time-based, leakage-free split — well above the ~0.50 a random classifier would score, confirming the pipeline produces a usable, non-trivial feature set. This is a deliberately conservative baseline (a linear model on a lightly-engineered feature set); the Data Science track's Week 5 modelling work is expected to improve on this materially, and this number exists specifically to give that work a concrete floor to beat.


In [11]:
model_path = model.save()
print(f"Model artifact saved to: {model_path}")

Model artifact saved to: /home/claude/healthconnect-repo/models/baseline_logistic_regression.joblib


## 8. Testing Evidence

22 automated tests cover every module in `src/` (ingestion, validation, feature engineering, and the model wrapper) using small, hand-crafted fixture data rather than the full dataset, so they run in about a second and pin down exact expected behaviour — including a regression test for a real bug caught during this week's implementation (see `tests/test_feature_engineering.py`, and Section 9 below).

Run from the repository root with:
```bash
python -m pytest tests/ -v
```
Full output saved to `test_results.txt` in the repository root.


## 9. Implementation Issues Encountered

Documenting these because the brief asks for issues encountered, not just successes:

1. **Double-drop bug:** `appointment_outcome` was initially listed in `config.LEAKAGE_COLUMNS` for documentation purposes, but that list is also used to *drop* columns — causing a `KeyError` when the pipeline tried to drop it a second time later. Fixed by removing it from that list and documenting the target column separately in `config.py`, with a regression test added (`tests/test_feature_engineering.py::test_drop_leakage_columns_removes_waiting_time`) to prevent a repeat.
2. **Missed categorical column:** `booking_lead_bucket` is engineered *after* the original categorical column list was defined, so it was initially left un-encoded and reached the model as a category dtype, raising a `Cannot cast str dtype to float64` error at `.fit()` time. Fixed by adding it to `config.CATEGORICAL_COLUMNS` with a comment explaining why engineered columns need to be added to that list too.
3. **Historical field inconsistency (data limitation, not a code bug):** discovered during validation, not anticipated in Week 4 — documented in Section 3.2 above and in the accompanying Pipeline Documentation report.


## 10. Summary & Next Steps

This notebook demonstrates a working, tested, end-to-end pipeline: raw CSV → validated → leakage-free, point-in-time features → time-based split → trained baseline model → saved artifact. The reusable logic lives in `src/`, covered by 22 passing tests, ready for the Data Science track to build their modelling work on top of via the same `NoShowRiskModel` interface.

**Proposed focus for Week 6:** integrate the Data Science track's finalised feature/target decisions once available, add a lightweight batch-scoring script (Section 5 of the Week 4 architecture) that scores a set of upcoming appointments using a saved model artifact, and begin drafting the monitoring/drift-check component.
